In [19]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [20]:
import logging
import pandas as pd
import delta_sharing

import general_functions.databricks_client as db_client
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import send_to_innkeepr_api_paginated, call_api_with_accountId

In [21]:
customer = "ahead-nutrition.com" #6617da9c01c2ab3bd12c63cb
date = "2026-05-07"#T20:00:00.000Z"
end_date = "2026-05-08"#T27T00:00:00.000Z"
use_session_id = False
url = return_api_url()
print(f"url = {url}")
workspaces = return_workspace_ids()
workspace_id = [acc["id"] for acc in workspaces if acc["name"] == customer]
try:
    workspace_id = workspace_id[0]
except IndexError:
    names = [acc["name"] for acc in workspaces]
    raise KeyError(f"Customer {customer} not found in {names}")

In [22]:
google_param = call_api_with_accountId(
    f"{url}/connections/query",
    workspace_id,
    {"name": "googleAdwords"},
    logging
)
google_param = google_param[0]["options"]["urlCampaignParam"]
google_param

In [23]:
meta_param = call_api_with_accountId(
    f"{url}/connections/query",
    workspace_id,
    {"name": "facebook"},
    logging
)
print(meta_param)
meta_param = meta_param[0]["options"]["urlTrackingParam"]
meta_param

In [24]:
if use_session_id:
    profile_path = db_client.return_databricks_client()
    table_path = f"{profile_path}#delta_share_events.{workspace_id}.features_view_30_outlook"
    df = delta_sharing.load_as_pandas(table_path, limit=500000)
    df = df[df["created"].astype("string")>=date]
    print(df["created"].min(), df["created"].max())
    session_ids=df["session"].unique().tolist()
    print(f"len session_ids = {len(session_ids)}")
    content = {"sessionId":session_ids}
else:
    content = {"created":{"$gte":date, "$lt":end_date}}

In [25]:
sessions_response = send_to_innkeepr_api_paginated( 
    f"{url}/sessions/query",
    workspace_id,
    content,
    logging
)

In [26]:
sessions = pd.json_normalize(sessions_response)
sessions

In [27]:
# google
google_external_ids = sessions[f"campaign.{google_param}"].dropna().unique().tolist()
google_signals = send_to_innkeepr_api_paginated(
    f"{url}/signals/query",
    workspace_id,
    {"externalId":google_external_ids},
    logging
)
google_signals = pd.json_normalize(google_signals)
google_signals

In [28]:
google_signals_gclid = send_to_innkeepr_api_paginated(
    f"{url}/signals/query",
    workspace_id,
    {"externalId":sessions["campaign.gclid"].dropna().unique().tolist()},
    logging
)
google_signals_gclid = pd.json_normalize(google_signals_gclid)
google_signals_gclid

In [29]:
google_signals = pd.concat([google_signals, google_signals_gclid])

In [30]:
google_treatments = send_to_innkeepr_api_paginated(
    f"{url}/treatments/query",
    workspace_id,
    {"id":google_signals["relates_to.treatment"].dropna().unique().tolist()},
    logging
)
google_treatments = pd.json_normalize(google_treatments)
google_treatments

In [31]:
facebook_external_ids = sessions[f"campaign.{meta_param}"].dropna().unique().tolist()
print(len(facebook_external_ids))
facebook_signals = send_to_innkeepr_api_paginated(
    f"{url}/signals/query",
    workspace_id,
    {"externalId":facebook_external_ids},
    logging
)
facebook_signals = pd.json_normalize(facebook_signals)
facebook_signals

In [32]:
facebook_treatments = send_to_innkeepr_api_paginated(
    f"{url}/treatments/query",
    workspace_id,
    {"id":facebook_signals["relates_to.treatment"].dropna().unique().tolist()},
    logging
)
facebook_treatments = pd.json_normalize(facebook_treatments)
facebook_treatments